Régression avec le critère 2SD et 1SD pour détecter les vendeurs algorithmiques

## Présentation du modèle

Nous traitons d'un marché avec des prix quasiment homogènes, nous nous attenderions donc à ce que les vendeurs fixent leur prix au coût marginal. 

La littérature soutient que les algorithmes de tarification engendrent des comportements collusifs et des prix plus élevés. _"The high prices are sustained by collusive strategies with a finite phase of punishment followed by a gradual return to cooperation"_ (Artificial Intelligence, Algorithmic Pricing, and Collusion
Emilio Calvano and al). Chen et al dans _An Empirical Analysis of Algorithmic Pricing on Amazon Marketplace_  concluent que la tarification algorithmique engendre des prix plus élevés sur la marketplace d'Amazon. 

Nous voulons tester si sur les prix des iPhone Leclerc, ces constats sont aussi vérifiés ou si comme dans l'article d'Hanspacha et al nous ne trouvons pas d'évidences de causalité entre algorithmes de tarification et des prix plus élevés.

## Buy Box selon Leclerc

Le **BboxPrice**, selon les critères de Leclerc, privilégiera toujours un produit de la marque Leclerc lorsqu'il existe un article correspondant dans la même catégorie que celui vendu. À défaut, il sélectionnera le produit d’un autre vendeur en se basant principalement sur le prix et l'état du téléphone, ainsi que sur d'autres critères.


Dans nos données, le BboxPrice est le premier produit affiché pour chaque  temps _t_ et pour chaque produit _i_.


$$
\log(\text{BboxPrice}_{i,t}) 
= \beta_0 
+ \beta_1 \,\text{Leclerc}_{i,t} 
+ \beta_2 \,\text{N.Alg}_{i,t}
+ \mathbf{X}
+ \mu_{i} \times \lambda_{t} 
+ \varepsilon_{i,t}
\quad (1)
$$

- $\text{Leclerc}_{i,t} $ indique si l'opérateur de la plateforme est en concurrence en tant que vendeur pour le produit concerné.

- $\text{N.Alg}_{i,t}$ est un ensemble de variables indicatrices (dummy) qui comptent le nombre de vendeurs algorithmiques pour le produit 𝑖 à l’instant 𝑡, selon les critères 1SD. 

-  **𝑋** vecteurs de variables de contrôle. Celui-ci inclut notamment _shippingDays_, le délai de livraison d’un produit en jours, et _Seller Rating_, la note du vendeur sur une échelle de 0 à 5.

- $\mu_{i}$ capture l'effet fixe spécifique au produit 
- $\lambda_{t}$ capture l'effet fixe spécifique au temps.

### Correction de l'endogénéité

Il existe des sources possibles d'endogénéité:

- **Endogénéité dans l'adoption du logiciel** :
Un vendeur doit décider d'acquérir un logiciel tiers pour pouvoir adopter une tarification algorithmique. Cette décision d'adoption peut dépendre de caractéristiques (observables ou non) qui influencent également le comportement des prix.
- **Endogénéité dans l'utilisation du logiciel** :
Même si le vendeur possède le logiciel nécessaire, il peut choisir, pour chaque produit, de l'utiliser ou non. Cette décision peut varier selon le produit et le moment, et peut être corrélée avec des facteurs non observés spécifiques à un produit à un instant donné.

Pour atténuer ce problème, une approche courante consiste à inclure **des effets fixes**. Dans le cas présent, nous intégrons des effets fixes produit-jour ($\mu_{i} \times \lambda_{t}$). 

Cela permet de contrôler pour toutes les caractéristiques inobservées qui varient à la fois par produit et par jour, et ainsi d'améliorer l'estimation des effets de la tarification algorithmique.

**$\mu_{i}$** absorbe toutes les caractéristiques inobservées qui ne varient pas dans le temps pour chaque produit, et 
**$\lambda_{t}$** contrôle les chocs ou tendances communs à tous les produits à un moment donné.

In [108]:
import pandas as pd

df = pd.read_csv('../Detection algo/data_algo1.csv')

Pour chaque jeu de données récupéré, le premier scrappé est le produit de la BBox.

In [109]:
bbox = pd.read_csv('../Formatage/BBox.csv')

Les données ont été scrappées 2157 fois entre le 24 décembre 2024 et le 11 février 2025 avec un lapse de temps moyen de 33 minutes entre chaque scrapping. Nous avons été bloqués sur la période de Noël.

In [110]:
# Liste des colonnes à sommer
cols = [
    'is_algorithmic_1sd', 
    'is_algorithmic_1sd_always', 
    'is_algorithmic_2sd', 
    'is_algorithmic_2sd_always', 
    'shipping_days', 
    'Seller Rating', 
    'NbSellerRatings', 
    'OCCASION - BON ÉTAT', 
    'OCCASION - EXCELLENT ÉTAT', 
    'OCCASION - PARFAIT - JAMAIS UTILISÉ',
    'OCCASION - TRÉS BON ÉTAT', 
    'OCCASION - ÉTAT CORRECT', 
    'Espagne', 
    'Italie', 
    'Lettonie', 
    'Luxembourg',
    'Leclerc'
]

# Regrouper par Timestamp et Product Name puis sommer les colonnes indiquées
df_grouped = df.groupby(['Timestamp', 'Product Name'])[cols].sum().reset_index()

Dans 88% des cas la Buy Box est détenue par Leclerc

In [111]:
sum(df_grouped['Leclerc'])/len(df_grouped)

0.8833583208395802

Ajout du prix de la BuyBox aux données et transformation logarithmique de la variable

In [112]:
import numpy as np
bbox['BBox_Price'] = np.log10(bbox['Price'])

In [113]:
df_final = df_grouped.merge(bbox[['BBox_Price','Timestamp', 'Product Name']], on=['Timestamp', 'Product Name'], how='left')

In [114]:
df_final.isna().sum()

Timestamp                              0
Product Name                           0
is_algorithmic_1sd                     0
is_algorithmic_1sd_always              0
is_algorithmic_2sd                     0
is_algorithmic_2sd_always              0
shipping_days                          0
Seller Rating                          0
NbSellerRatings                        0
OCCASION - BON ÉTAT                    0
OCCASION - EXCELLENT ÉTAT              0
OCCASION - PARFAIT - JAMAIS UTILISÉ    0
OCCASION - TRÉS BON ÉTAT               0
OCCASION - ÉTAT CORRECT                0
Espagne                                0
Italie                                 0
Lettonie                               0
Luxembourg                             0
Leclerc                                0
BBox_Price                             0
dtype: int64

### Régression1 : Avec seuil 1SD

In [115]:
import statsmodels.formula.api as smf

df_final['Timestamp'] = pd.to_datetime(df_final['Timestamp'])
df_final['date'] = df_final['Timestamp'].dt.date
# Utiliser cette variable pour l'effet fixe temporel
model = smf.ols(
    "BBox_Price ~ is_algorithmic_1sd + shipping_days + Q('Seller Rating') + NbSellerRatings + "
    "Q('OCCASION - BON ÉTAT') + Q('OCCASION - EXCELLENT ÉTAT') + Q('OCCASION - PARFAIT - JAMAIS UTILISÉ') + "
    "Q('OCCASION - TRÉS BON ÉTAT') + Q('OCCASION - ÉTAT CORRECT') + Espagne + Italie + Lettonie + Luxembourg + "
    "Leclerc + (C(Q('Product Name')) * C(date))",
    data=df_final
).fit()


print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             BBox_Price   R-squared:                       0.972
Model:                            OLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     1603.
Date:                Sun, 09 Mar 2025   Prob (F-statistic):               0.00
Time:                        15:23:00   Log-Likelihood:                 60672.
No. Observations:               23345   AIC:                        -1.203e+05
Df Residuals:                   22841   BIC:                        -1.163e+05
Df Model:                         503                                         
Covariance Type:            nonrobust                                         
                                                                                                                                                   coef    std err          t      P>|t|      [0.025      0.975]
-

**Résultats:**

| Variables                          | coef       | p-value |
|------------------------------------|------------|--------|
| Intercept                          | 3.1492     | 0.000  |
| N.Algo                 | -0.0064    | 0.000  |
| shipping_days                      | -9.921e-05 | 0.000  |
| Seller Rating                      | -0.0002    | 0.000  |
| NbSellerRatings                    | -2.103e-05 | 0.000  |
| OCCASION - BON ÉTAT                | 0.0134     | 0.000  |
| OCCASION - EXCELLENT ÉTAT          | 0.0031     | 0.478  |
| OCCASION - PARFAIT - JAMAIS UTILISÉ | 0.0072     | 0.106  |
| OCCASION - TRÉS BON ÉTAT           | 0.0002     | 0.871  |
| OCCASION - ÉTAT CORRECT            | 0.0189     | 0.000  |
| Espagne                            | 0.0153     | 0.000  |
| Italie                             | 0.0059     | 0.000  |
| Lettonie                           | 0.0123     | 0.000  |
| Luxembourg                         | -0.0158    | 0.000  |
| Leclerc                            | -0.0405    | 0.000  |


### Régression 2 : Avec seuil 2SD

In [116]:
import statsmodels.formula.api as smf

df_final['Timestamp'] = pd.to_datetime(df_final['Timestamp'])
df_final['date'] = df_final['Timestamp'].dt.date
# Utiliser cette variable pour l'effet fixe temporel
model = smf.ols(
    "BBox_Price ~ is_algorithmic_2sd + shipping_days + Q('Seller Rating') + NbSellerRatings  + "
    "Q('OCCASION - BON ÉTAT') + Q('OCCASION - EXCELLENT ÉTAT') + Q('OCCASION - PARFAIT - JAMAIS UTILISÉ') + "
    "Q('OCCASION - TRÉS BON ÉTAT') + Q('OCCASION - ÉTAT CORRECT') + Espagne + Italie + Lettonie + Luxembourg + "
    "Leclerc + (C(Q('Product Name')) * C(date))",
    data=df_final
).fit()


print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             BBox_Price   R-squared:                       0.972
Model:                            OLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     1597.
Date:                Sun, 09 Mar 2025   Prob (F-statistic):               0.00
Time:                        15:23:05   Log-Likelihood:                 60631.
No. Observations:               23345   AIC:                        -1.203e+05
Df Residuals:                   22841   BIC:                        -1.162e+05
Df Model:                         503                                         
Covariance Type:            nonrobust                                         
                                                                                                                                                   coef    std err          t      P>|t|      [0.025      0.975]
-

**Résultats:**

| Variables                          | coef       |p-value |
|------------------------------------|------------|--------|
| Intercept                          | 3.1467     | 0.000  |
| N.Algo                | 0.0019     | 0.171  |
| shipping_days                      | -9.745e-05 | 0.000  |
| Seller Rating                      | -0.0004    | 0.000  |
| NbSellerRatings                    | -1.893e-05 | 0.000  |
| OCCASION - BON ÉTAT                | 0.0077     | 0.000  |
| OCCASION - EXCELLENT ÉTAT          | -0.0061    | 0.188  |
| OCCASION - PARFAIT - JAMAIS UTILISÉ | 0.0070     | 0.117  |
| OCCASION - TRÉS BON ÉTAT           | -0.0065    | 0.000  |
| OCCASION - ÉTAT CORRECT            | 0.0191     | 0.000  |
| Espagne                            | 0.0156     | 0.000  |
| Italie                             | 0.0053     | 0.000  |
| Lettonie                           | 0.0131     | 0.000  |
| Luxembourg                         | -0.0147    | 0.000  |
| Leclerc                            | -0.0405    | 0.000  |


### Régression 3 : Avec seuil de 1 SD et qui classe tout vendeur algorithmique comme algorithmique pour tous ses produits

In [117]:
import statsmodels.formula.api as smf

df_final['Timestamp'] = pd.to_datetime(df_final['Timestamp'])
df_final['date'] = df_final['Timestamp'].dt.date
# Utiliser cette variable pour l'effet fixe temporel
model = smf.ols(
    "BBox_Price ~ is_algorithmic_1sd_always + shipping_days + Q('Seller Rating') + NbSellerRatings + "
    "Q('OCCASION - BON ÉTAT') + Q('OCCASION - EXCELLENT ÉTAT') + Q('OCCASION - PARFAIT - JAMAIS UTILISÉ') + "
    "Q('OCCASION - TRÉS BON ÉTAT') + Q('OCCASION - ÉTAT CORRECT') + Espagne + Italie + Lettonie + Luxembourg + "
    "Leclerc + (C(Q('Product Name')) * C(date))",
    data=df_final
).fit()


print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             BBox_Price   R-squared:                       0.972
Model:                            OLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     1597.
Date:                Sun, 09 Mar 2025   Prob (F-statistic):               0.00
Time:                        15:23:10   Log-Likelihood:                 60630.
No. Observations:               23345   AIC:                        -1.203e+05
Df Residuals:                   22841   BIC:                        -1.162e+05
Df Model:                         503                                         
Covariance Type:            nonrobust                                         
                                                                                                                                                   coef    std err          t      P>|t|      [0.025      0.975]
-

**Résultats:**

| Variables                          | coef       |p-value |
|------------------------------------|------------|--------|
| Intercept                          | 3.1469     | 0.000  |
| N.Algo           | 0.0005     | 0.485  |
| shipping_days                      | -9.601e-05 | 0.000  |
| Seller Rating                      | -0.0004    | 0.000  |
| NbSellerRatings                    | -1.838e-05 | 0.000  |
| OCCASION - BON ÉTAT                | 0.0087     | 0.000  |
| OCCASION - EXCELLENT ÉTAT          | -0.0044    | 0.322  |
| OCCASION - PARFAIT - JAMAIS UTILISÉ | 0.0070     | 0.115  |
| OCCASION - TRÉS BON ÉTAT           | -0.0053    | 0.000  |
| OCCASION - ÉTAT CORRECT            | 0.0194     | 0.000  |
| Espagne                            | 0.0157     | 0.000  |
| Italie                             | 0.0050     | 0.000  |
| Lettonie                           | 0.0131     | 0.000  |
| Luxembourg                         | -0.0146    | 0.000  |
| Leclerc                            | -0.0405    | 0.000  |


### Régression 4 : Avec seuil de 2 SD et qui classe tout vendeur algorithmique comme algorithmique pour tous ses produits

In [118]:
import statsmodels.formula.api as smf

df_final['Timestamp'] = pd.to_datetime(df_final['Timestamp'])
df_final['date'] = df_final['Timestamp'].dt.date
# Utiliser cette variable pour l'effet fixe temporel
model = smf.ols(
    "BBox_Price ~ is_algorithmic_2sd_always + shipping_days + Q('Seller Rating') + NbSellerRatings  + "
    "Q('OCCASION - BON ÉTAT') + Q('OCCASION - EXCELLENT ÉTAT') + Q('OCCASION - PARFAIT - JAMAIS UTILISÉ') + "
    "Q('OCCASION - TRÉS BON ÉTAT') + Q('OCCASION - ÉTAT CORRECT') + Espagne + Italie + Lettonie + Luxembourg + "
    "Leclerc + (C(Q('Product Name')) * C(date))",
    data=df_final
).fit()


print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             BBox_Price   R-squared:                       0.972
Model:                            OLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     1600.
Date:                Sun, 09 Mar 2025   Prob (F-statistic):               0.00
Time:                        15:23:14   Log-Likelihood:                 60630.
No. Observations:               23345   AIC:                        -1.203e+05
Df Residuals:                   22842   BIC:                        -1.162e+05
Df Model:                         502                                         
Covariance Type:            nonrobust                                         
                                                                                                                                                   coef    std err          t      P>|t|      [0.025      0.975]
-

**Résultats:**

| Variables                          | coef       | p-value |
|------------------------------------|------------|--------|
| Intercept                          | 3.1467     | 0.000  |
| N.Algo                             | 6.563e-05  | 0.954  |
| shipping_days                      | -9.687e-05 | 0.000  |
| Seller Rating                      | -0.0004    | 0.000  |
| NbSellerRatings                    | -1.87e-05  | 0.000  |
| OCCASION - BON ÉTAT                | 0.0090     | 0.000  |
| OCCASION - EXCELLENT ÉTAT          | -0.0041    | 0.216  |
| OCCASION - PARFAIT - JAMAIS UTILISÉ | 0.0070     | 0.116  |
| OCCASION - TRÉS BON ÉTAT           | -0.0049    | 0.000  |
| OCCASION - ÉTAT CORRECT            | 0.0194     | 0.000  |
| Espagne                            | 0.0156     | 0.000  |
| Italie                             | 0.0053     | 0.000  |
| Lettonie                           | 0.0130     | 0.000  |
| Luxembourg                         | -0.0147    | 0.000  |
| Leclerc                            | -0.0405    | 0.000  |


In [119]:
len(df_final['Product Name'].unique())

10